# Origination Scorecard — Freddie Mac SFLLD (2015–2019)

This notebook builds a **classic points-based origination scorecard** using Freddie Mac Single-Family Loan-Level Dataset (SFLLD) data stored in BigQuery.

**Pipeline overview:**
1. Pull origination + performance data from BigQuery (2015–2019 origination vintages)
2. Define *bad* flag — 90+ DPD within 24 months of origination or foreclosure / REO
3. WoE binning + Information Value (IV) feature selection
4. Logistic Regression training on WoE-transformed features
5. Standard scorecard scaling (PDO = 20, base score = 600, base odds = 1:30)
6. Validation: KS, Gini, ROC-AUC, score distribution by good/bad

## 1. Install and Import Required Libraries

In [ ]:
import subprocess, sys

PKGS = [
    "google-cloud-bigquery",
    "google-cloud-bigquery-storage",
    "pyarrow",
    "db-dtypes",
    "pandas",
    "numpy",
    "scikit-learn",
    "scipy",
    "matplotlib",
    "seaborn",
    "joblib",
    "tqdm",
    "optbinning",   # WoE / IV binning
]

for pkg in PKGS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages installed.")

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from optbinning import BinningProcess, Scorecard, OptimalBinning

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

PROJECT_ROOT = Path("..").resolve()
MODEL_OUT = PROJECT_ROOT / "models" / "credit_risk"
MODEL_OUT.mkdir(parents=True, exist_ok=True)

print("Imports OK  |  model output →", MODEL_OUT)

## 2. Connect to BigQuery and Extract Data

We pull **origination** attributes and derive the *bad* outcome from the **performance** table:
- **Bad flag** = 1 if the loan ever reached 90+ DPD *or* had a foreclosure/REO zero-balance event within **24 months** of the first payment date.
- We restrict origination vintages to **2015–2019** (first_payment_date 201501–201912) so every loan has at least 24 months of observable performance.

In [ ]:
import google.auth
from google.oauth2 import service_account
from google.cloud import bigquery

# ─── Credentials ────────────────────────────────────────────────────────────
SA_KEY = PROJECT_ROOT / ".gcp-sa-key-dev.json"
GCP_PROJECT = "ai-risk-workflow"
BQ_DATASET  = "freddie_mac_sflld"

if SA_KEY.exists():
    creds = service_account.Credentials.from_service_account_file(
        str(SA_KEY),
        scopes=["https://www.googleapis.com/auth/cloud-platform"],
    )
    client = bigquery.Client(project=GCP_PROJECT, credentials=creds)
    print(f"Authenticated via SA key: {SA_KEY.name}")
else:
    # Fall back to Application Default Credentials
    creds, project = google.auth.default()
    client = bigquery.Client(project=GCP_PROJECT, credentials=creds)
    print("Authenticated via ADC")

In [ ]:
BQ_QUERY = f"""
-- ============================================================
-- Origination Scorecard — labelled dataset (2015-2019 vintages)
-- ============================================================
-- BAD = 90+ DPD within 24 months of origination
--       OR zero_balance_code in ('02','03','06','09') within 24 months
--       (02=Third-Party Sale, 03=Short Sale, 06=Repurchase, 09=Deed-in-Lieu/Foreclosure)
-- ============================================================
WITH orig AS (
    SELECT
        loan_sequence_number,
        credit_score,
        first_payment_date,
        first_time_homebuyer_flag,
        maturity_date,
        mi_pct,
        number_of_units,
        occupancy_status,
        ocltv,
        odti,
        original_upb,
        oltv,
        original_interest_rate,
        channel,
        product_type,
        property_state,
        property_type,
        loan_purpose,
        original_loan_term,
        number_of_borrowers,
        super_conforming_flag,
        io_indicator
    FROM `{GCP_PROJECT}.{BQ_DATASET}.freddie_origination`
    -- vintages 2015-01 through 2019-12
    WHERE first_payment_date BETWEEN 201501 AND 201912
),
perf_bad AS (
    SELECT
        p.loan_sequence_number,
        MAX(
            CASE
                WHEN SAFE_CAST(p.current_delinquency_status AS INT64) >= 3
                     AND p.loan_age BETWEEN 1 AND 24
                THEN 1
                WHEN p.zero_balance_code IN ('02','03','06','09')
                     AND p.loan_age BETWEEN 1 AND 24
                THEN 1
                ELSE 0
            END
        ) AS bad_flag
    FROM `{GCP_PROJECT}.{BQ_DATASET}.freddie_performance` p
    GROUP BY p.loan_sequence_number
)
SELECT
    o.*,
    COALESCE(pb.bad_flag, 0) AS bad_flag
FROM orig o
LEFT JOIN perf_bad pb USING (loan_sequence_number)
"""

print("Running BQ query …")
df_raw = client.query(BQ_QUERY).to_dataframe()
print(f"Rows fetched: {len(df_raw):,}  |  Columns: {list(df_raw.columns)}")
print(f"Bad rate: {df_raw['bad_flag'].mean():.2%}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Basic overview ──────────────────────────────────────────────────────────
print("=== Shape ===")
print(f"  Rows: {len(df_raw):,}   Columns: {df_raw.shape[1]}")

print("\n=== Bad flag distribution ===")
vc = df_raw["bad_flag"].value_counts()
print(f"  Good (0): {vc.get(0,0):>10,}  ({vc.get(0,0)/len(df_raw):.2%})")
print(f"  Bad  (1): {vc.get(1,0):>10,}  ({vc.get(1,0)/len(df_raw):.2%})")

print("\n=== Vintage breakdown ===")
df_raw["vintage_year"] = (df_raw["first_payment_date"] // 100).astype(int)
print(df_raw.groupby("vintage_year")["bad_flag"].agg(["count","sum","mean"]).rename(
    columns={"count":"Loans", "sum":"Bads", "mean":"Bad Rate"}
).to_string())

print("\n=== Missing values (%) ===")
miss = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
print(miss[miss > 0].to_string())

In [ ]:
# ── Numeric feature distributions by bad flag ───────────────────────────────
NUMERIC_FEATURES = [
    "credit_score", "oltv", "ocltv", "odti",
    "original_interest_rate", "original_upb", "original_loan_term",
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURES):
    ax = axes[i]
    good = df_raw.loc[df_raw["bad_flag"] == 0, col].dropna()
    bad  = df_raw.loc[df_raw["bad_flag"] == 1, col].dropna()
    ax.hist(good, bins=40, alpha=0.5, color="steelblue", label="Good", density=True)
    ax.hist(bad,  bins=40, alpha=0.5, color="crimson",   label="Bad",  density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
fig.suptitle("Feature Distributions — Good vs Bad", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# ── Bad rate by categorical variables ───────────────────────────────────────
CAT_FEATURES = [
    "occupancy_status", "loan_purpose", "channel",
    "property_type", "first_time_homebuyer_flag", "number_of_units",
]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
axes = axes.flatten()

for i, col in enumerate(CAT_FEATURES):
    ax = axes[i]
    grp = df_raw.groupby(col)["bad_flag"].agg(["mean", "count"]).reset_index()
    grp = grp.sort_values("mean", ascending=False)
    bars = ax.bar(grp[col].astype(str), grp["mean"] * 100, color="salmon", edgecolor="black")
    ax.set_title(f"Bad rate by {col}", fontsize=10)
    ax.set_ylabel("Bad rate (%)")
    ax.set_ylim(0, grp["mean"].max() * 130)
    for bar, (_, row) in zip(bars, grp.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.1,
            f"n={row['count']:,.0f}",
            ha="center", va="bottom", fontsize=7, rotation=45
        )

fig.suptitle("Bad Rate by Categorical Feature", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

## 4. Data Preprocessing and Feature Engineering

In [ ]:
# ── Feature selection for scorecard ─────────────────────────────────────────
# Origination scorecard uses only information available at time of application.
SCORECARD_FEATURES_NUM = [
    "credit_score",           # FICO
    "oltv",                   # Original LTV
    "odti",                   # Original DTI
    "original_interest_rate", # Note rate (proxy for risk/pricing tier)
    "original_upb",           # Loan amount
    "original_loan_term",     # Term (months)
    "mi_pct",                 # Mortgage insurance %
]

SCORECARD_FEATURES_CAT = [
    "occupancy_status",           # P/I/S (primary / investment / second)
    "loan_purpose",               # P=purchase, C=cash-out, N=rate-term
    "channel",                    # R/B/C/T
    "property_type",              # SF/PU/CO/MH
    "first_time_homebuyer_flag",  # Y/N/9
    "number_of_units",            # 1-4
    "io_indicator",               # Interest-only flag
    "super_conforming_flag",      # Y/N
]

ALL_FEATURES = SCORECARD_FEATURES_NUM + SCORECARD_FEATURES_CAT
TARGET = "bad_flag"

df = df_raw[ALL_FEATURES + [TARGET]].copy()

# ── Imputation ────────────────────────────────────────────────────────────────
# Numeric: median imputation
for col in SCORECARD_FEATURES_NUM:
    med = df[col].median()
    df[col] = df[col].fillna(med)

# Categorical: mode imputation (or 'UNK')
for col in SCORECARD_FEATURES_CAT:
    df[col] = df[col].fillna("UNK").astype(str).str.strip().str.upper()

# ── Clip extreme outliers (Winsorize at 1st/99th pct) ────────────────────────
for col in SCORECARD_FEATURES_NUM:
    lo, hi = df[col].quantile([0.01, 0.99])
    df[col] = df[col].clip(lo, hi)

print("Preprocessing complete.")
print(df.shape)
print(df[TARGET].value_counts())

In [ ]:
# ── Train / Validation split (70/30, stratified by vintage + bad_flag) ────────
df["vintage_year"] = (df_raw["first_payment_date"] // 100).astype(int)

# Temporal split: train on 2015-2017, validate on 2018-2019
df_train = df[df["vintage_year"] <= 2017].drop(columns="vintage_year").reset_index(drop=True)
df_val   = df[df["vintage_year"] >= 2018].drop(columns="vintage_year").reset_index(drop=True)

X_train, y_train = df_train[ALL_FEATURES], df_train[TARGET]
X_val,   y_val   = df_val[ALL_FEATURES],   df_val[TARGET]

print(f"Train: {len(X_train):,} rows  bad rate={y_train.mean():.2%}")
print(f"Val  : {len(X_val):,}   rows  bad rate={y_val.mean():.2%}")

## 5. Weight of Evidence (WoE) and Information Value (IV)

$$WoE_i = \ln\!\left(\frac{P(\text{Events in bin } i)}{P(\text{Non-Events in bin } i)}\right)$$

$$IV = \sum_i \bigl(D_{E,i} - D_{NE,i}\bigr) \times WoE_i$$

| IV range | Predictive power |
|---|---|
| < 0.02 | Useless |
| 0.02 – 0.1 | Weak |
| 0.1 – 0.3 | Medium |
| 0.3 – 0.5 | Strong |
| > 0.5 | Suspicious (check for data leakage) |

In [ ]:
# ── Compute WoE / IV using optbinning ────────────────────────────────────────
variable_names = ALL_FEATURES

dtype_map = {c: "numerical" for c in SCORECARD_FEATURES_NUM}
dtype_map.update({c: "categorical" for c in SCORECARD_FEATURES_CAT})

binning_fit_params = {
    c: {"max_n_prebins": 20, "min_prebin_size": 0.02} if dtype_map[c] == "numerical"
    else {}
    for c in variable_names
}

bp = BinningProcess(
    variable_names=variable_names,
    categorical_variables=SCORECARD_FEATURES_CAT,
    binning_fit_params={c: {} for c in variable_names},  # defaults per variable
    max_n_prebins=20,
    min_prebin_size=0.02,
)

print("Fitting BinningProcess …")
bp.fit(X_train, y_train)
print("Done.")

# Summary table
bp_summary = bp.summary()
iv_table = (
    bp_summary[["Name", "IV"]]
    .rename(columns={"Name": "feature", "IV": "iv"})
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)

def iv_label(iv):
    if iv < 0.02: return "Useless"
    if iv < 0.10: return "Weak"
    if iv < 0.30: return "Medium"
    if iv < 0.50: return "Strong"
    return "Suspicious"

iv_table["strength"] = iv_table["iv"].apply(iv_label)
print(iv_table.to_string(index=False))

In [ ]:
# ── IV bar chart ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
colors = iv_table["iv"].apply(
    lambda v: "steelblue" if v >= 0.10 else ("orange" if v >= 0.02 else "lightgray")
)
ax.barh(iv_table["feature"][::-1], iv_table["iv"][::-1], color=colors.values[::-1])
ax.axvline(0.1, color="green", linestyle="--", linewidth=1.2, label="IV = 0.10 (medium threshold)")
ax.set_xlabel("Information Value (IV)")
ax.set_title("Feature IV — Origination Scorecard", fontsize=13, fontweight="bold")
ax.legend()
fig.tight_layout()
plt.show()

# ── Select features with IV ≥ 0.02 ──────────────────────────────────────────
SELECTED = iv_table.loc[iv_table["iv"] >= 0.02, "feature"].tolist()
print(f"\nSelected {len(SELECTED)} features (IV ≥ 0.02):  {SELECTED}")

In [ ]:
# ── Detailed WoE table for top features ──────────────────────────────────────
for feat in SELECTED[:4]:
    ob = bp.get_binned_variable(feat)
    woe_df = ob.binning_table.build()
    print(f"\n{'='*55}")
    print(f"  WoE table: {feat}")
    print(f"{'='*55}")
    print(woe_df[["Bin","Count","Count (%)","Event rate","WoE","IV"]].to_string(index=False))

## 6. Logistic Regression Model Training

$$P(Y=1 \mid \mathbf{X}) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \cdots + \beta_n X_n)}}$$

We train on WoE-transformed features from the selected variable set.

In [ ]:
# ── Re-fit binning process on selected features only ─────────────────────────
bp_sel = BinningProcess(
    variable_names=SELECTED,
    categorical_variables=[c for c in SCORECARD_FEATURES_CAT if c in SELECTED],
    max_n_prebins=20,
    min_prebin_size=0.02,
)

# ── Scorecard (Logistic Regression + WoE binning) ─────────────────────────────
# Standard PDO / base score / base odds settings
PDO        = 20      # Points to Double the Odds
BASE_SCORE = 600     # Score at base odds
BASE_ODDS  = 30      # Non-event : event odds at base score  (30:1 good:bad)

lr = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    C=0.1,            # light L2 regularisation
    class_weight="balanced",
    random_state=42,
)

scorecard = Scorecard(
    binning_process=bp_sel,
    estimator=lr,
    scaling_method="min_max",
    scaling_method_params={"min": 300, "max": 850},
    intercept_based=False,
    reverse_scorecard=True,      # higher score = lower risk (like FICO)
    pdo=PDO,
    odds=BASE_ODDS,
    scorecard_id="origination_v1",
)

print("Fitting scorecard model …")
scorecard.fit(X_train[SELECTED], y_train)
print("Done.")

In [ ]:
# ── Training set predictions ──────────────────────────────────────────────────
y_prob_train = scorecard.predict_proba(X_train[SELECTED])[:, 1]
y_score_train = scorecard.score(X_train[SELECTED])

auc_train = roc_auc_score(y_train, y_prob_train)
gini_train = 2 * auc_train - 1

print(f"Train  AUC  = {auc_train:.4f}   Gini = {gini_train:.4f}")

# Cross-validation (on train only using probability pipeline)
# Build WoE transform then LR manually for CV check
Xw_train = bp_sel.transform(X_train[SELECTED], metric="woe")
cv_auc = cross_val_score(
    LogisticRegression(solver="lbfgs", max_iter=500, C=0.1,
                       class_weight="balanced", random_state=42),
    Xw_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="roc_auc",
)
print(f"5-fold CV AUC  mean={cv_auc.mean():.4f}   std={cv_auc.std():.4f}")

## 7. Scorecard Scaling and Point Allocation

The raw log-odds from logistic regression are scaled to an integer points table:

$$\text{Score} = A - B \times \ln(\text{odds})$$

where  
$$B = \frac{PDO}{\ln 2}, \qquad A = \text{BaseScore} - B \times \ln(\text{BaseOdds})$$

Each characteristic (variable) contributes a points value for the bin the borrower falls into. The sum of all characteristic points equals the final score.

In [ ]:
# ── Points table ──────────────────────────────────────────────────────────────
points_df = scorecard.table(style=None)
print(points_df.to_string(index=False))

# Save to CSV
pts_path = MODEL_OUT / "origination_scorecard_points_v1.csv"
points_df.to_csv(pts_path, index=False)
print(f"\nPoints table saved → {pts_path}")

## 8. Scorecard Validation and Gini Coefficient

Key metrics on the **out-of-time** 2018–2019 holdout:

- **Gini**: $G = 2 \times AUC - 1$
- **KS statistic**: max separation between cumulative good/bad distributions
- **Score distribution**: histogram of scores for Good vs Bad accounts

In [ ]:
# ── Validation predictions ────────────────────────────────────────────────────
y_prob_val  = scorecard.predict_proba(X_val[SELECTED])[:, 1]
y_score_val = scorecard.score(X_val[SELECTED])

auc_val  = roc_auc_score(y_val, y_prob_val)
gini_val = 2 * auc_val - 1

# KS statistic
fpr, tpr, thresholds = roc_curve(y_val, y_prob_val)
ks_stat = float(np.max(np.abs(tpr - fpr)))

print("=" * 45)
print(f"  Validation Set (OOT 2018-2019)")
print("=" * 45)
print(f"  AUC    : {auc_val:.4f}")
print(f"  Gini   : {gini_val:.4f}")
print(f"  KS     : {ks_stat:.4f}")
print("=" * 45)

# Classification report at 50% PD threshold
y_pred_val = (y_prob_val >= 0.5).astype(int)
print("\nClassification Report (threshold = 0.50):")
print(classification_report(y_val, y_pred_val, target_names=["Good","Bad"]))

In [ ]:
# ── KS chart ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- KS curve ---
ax = axes[0]
good_prob = np.sort(y_prob_val[y_val == 0])
bad_prob  = np.sort(y_prob_val[y_val == 1])
thresholds_ks = np.linspace(0, 1, 300)
cum_good = np.array([np.mean(good_prob <= t) for t in thresholds_ks])
cum_bad  = np.array([np.mean(bad_prob  <= t) for t in thresholds_ks])
ks_idx   = np.argmax(np.abs(cum_good - cum_bad))

ax.plot(thresholds_ks, cum_bad,  color="crimson",   label="Bad (cumulative)")
ax.plot(thresholds_ks, cum_good, color="steelblue", label="Good (cumulative)")
ax.fill_between(thresholds_ks, cum_bad, cum_good, alpha=0.15, color="green")
ax.axvline(thresholds_ks[ks_idx], color="green", linestyle="--",
           label=f"KS = {ks_stat:.3f} @ {thresholds_ks[ks_idx]:.2f}")
ax.set_xlabel("Predicted probability of default")
ax.set_ylabel("Cumulative distribution")
ax.set_title("KS Chart — OOT Validation", fontsize=12, fontweight="bold")
ax.legend()

# --- ROC curve ---
ax2 = axes[1]
fpr_v, tpr_v, _ = roc_curve(y_val, y_prob_val)
ax2.plot(fpr_v, tpr_v, color="darkorange", lw=2, label=f"ROC (AUC={auc_val:.3f})")
ax2.plot([0,1], [0,1], "k--", lw=1)
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Curve — OOT Validation", fontsize=12, fontweight="bold")
ax2.legend()

plt.tight_layout()
plt.savefig(MODEL_OUT / "origination_scorecard_ks_roc.png", dpi=120)
plt.show()
print(f"Plot saved → {MODEL_OUT / 'origination_scorecard_ks_roc.png'}")

In [ ]:
# ── Score distribution: Good vs Bad ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
ax = axes[0]
good_scores = y_score_val[y_val == 0]
bad_scores  = y_score_val[y_val == 1]

ax.hist(good_scores, bins=40, alpha=0.55, color="steelblue", label="Good", density=True)
ax.hist(bad_scores,  bins=40, alpha=0.55, color="crimson",   label="Bad",  density=True)
ax.set_xlabel("Origination Score")
ax.set_ylabel("Density")
ax.set_title("Score Distribution — Good vs Bad", fontsize=12, fontweight="bold")
ax.legend()

# Score band bad rate table
ax2 = axes[1]
score_series = pd.Series(y_score_val, name="score")
bad_series   = pd.Series(y_val.values, name="bad_flag")
df_score = pd.concat([score_series, bad_series], axis=1)
df_score["band"] = pd.cut(df_score["score"],
                           bins=[300, 500, 550, 590, 620, 650, 680, 720, 850],
                           right=True)
band_tbl = df_score.groupby("band", observed=True)["bad_flag"].agg(
    Count="count", Bads="sum"
)
band_tbl["Goods"]    = band_tbl["Count"] - band_tbl["Bads"]
band_tbl["Bad Rate"] = band_tbl["Bads"] / band_tbl["Count"]

bars = ax2.bar(range(len(band_tbl)), band_tbl["Bad Rate"] * 100,
               color="salmon", edgecolor="black")
ax2.set_xticks(range(len(band_tbl)))
ax2.set_xticklabels([str(b) for b in band_tbl.index], rotation=35, ha="right", fontsize=8)
ax2.set_ylabel("Bad Rate (%)")
ax2.set_title("Bad Rate by Score Band", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nScore Band Report:")
print(band_tbl.to_string())

In [ ]:
# ── Save model artifacts ──────────────────────────────────────────────────────
MODEL_ARTIFACT = MODEL_OUT / "origination_scorecard_v1.pkl"
joblib.dump(scorecard, MODEL_ARTIFACT)
print(f"Scorecard model saved → {MODEL_ARTIFACT}")

# Save validation metrics
import json
metrics = {
    "model": "origination_scorecard_v1",
    "data": "freddie_mac_sflld 2015-2019",
    "train_vintages": "2015-2017",
    "val_vintages": "2018-2019",
    "selected_features": SELECTED,
    "pdo": PDO,
    "base_score": BASE_SCORE,
    "base_odds": BASE_ODDS,
    "train_auc":  round(float(auc_train),  4),
    "train_gini": round(float(gini_train), 4),
    "val_auc":    round(float(auc_val),    4),
    "val_gini":   round(float(gini_val),   4),
    "val_ks":     round(float(ks_stat),    4),
}
metrics_path = MODEL_OUT / "origination_scorecard_v1_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved      → {metrics_path}")

print("\n=== Final Model Summary ===")
for k, v in metrics.items():
    print(f"  {k:<22}: {v}")

## Summary

| Artifact | Location |
|---|---|
| Scorecard model | `models/credit_risk/origination_scorecard_v1.pkl` |
| Points table CSV | `models/credit_risk/origination_scorecard_points_v1.csv` |
| KS / ROC plot    | `models/credit_risk/origination_scorecard_ks_roc.png` |
| Metrics JSON     | `models/credit_risk/origination_scorecard_v1_metrics.json` |

### Next steps
- **Governance review** — PSI monitoring on new vintages, model validation package
- **Cut-off policy** — map score bands to Approve / Review / Decline tiers  
- **Champion/challenger** — compare vs LightGBM-based model  
- **Fair lending** — run disparate impact analysis on protected class segments  
- **Production** — wrap `scorecard.score()` in the decision-api endpoint